# 🚀 Batch Multi-File Shorts Generator (Kaggle GPU Edition)
**Upload MULTIPLE Videos → Sequential Processing → Cooldown → Zero GPU Waste!**

### Features:
- 📁 **Multi-File Upload**: Drag & drop all your videos at once
- ⏱️ **Smart Cooldown**: 30-60s GPU cooldown between files (configurable)
- 🛡️ **Session KeepAlive**: Auto-heartbeat every 45s prevents Kaggle timeout
- 🎙️ **SeedVC Voice Cloning**: Bulletproof custom voice conversion (2x fast mode)
- 🎵 **Auto BGM**: Smart best-drop detection applied to each video
- ✨ **4K Upscaling**: Real-ESRGAN on every file
- 🎬 **Auto-Transitions**: Scene detection + cinematic crossfades
- 📎 **Outro Integration**: Outro padding, scaling, fps matching & voice cloning

### ⚠️ Setup:
1. Enable **GPU T4 x2** (Settings → Accelerator → GPU T4 x2)
2. Enable **Internet** (Settings → Internet → On)
3. Run all cells in order!


In [ ]:
import os, gc, psutil, time, threading

print('=== 1. Environment Setup & Session Guardian ===')
os.system('echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# === Session Guardian (KeepAlive Heartbeat) ===
_guardian_start = time.time()
_guardian_active = True
def _session_guardian():
    _hb = 0
    while _guardian_active:
        time.sleep(45)
        _hb += 1
        mins = int((time.time() - _guardian_start) / 60)
        try:
            cpu = psutil.cpu_percent()
            ram = psutil.virtual_memory().percent
        except:
            cpu = ram = 0
        try:
            import subprocess
            nv = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader,nounits'],
                text=True).strip().split(', ')
            gpu_str = f"GPU VRAM: {float(nv[0])/1024:.1f}/{float(nv[1])/1024:.1f}GB"
        except:
            gpu_str = "GPU: Active"
        print(f"🟢 KeepAlive #{_hb} ({mins}m elapsed) | CPU:{cpu}% | RAM:{ram}% | {gpu_str}")

threading.Thread(target=_session_guardian, daemon=True).start()
print("🛡️ Session KeepAlive Engine started (45s heartbeat)")

print('Installing dependencies...')
!pip uninstall -y torchao
!pip install -q --upgrade "scipy>=1.13.0"
!pip install -q "transformers==5.19.0"
!pip install -q scenedetect moviepy==1.0.3 gradio soundfile audio-separator[gpu] ffmpeg-python kaggle
!pip install -q --no-build-isolation facexlib gfpgan
!pip uninstall -y torchao
!pip install -q "transformers==5.19.0"

print('✅ Dependencies installed!')

In [ ]:
print('=== 2. Setting up Real-ESRGAN & BasicSR (CUDA Acceleration) ===')
import subprocess, sys, os

# 1. Setup BasicSR cleanly from source with degradations patch (bypassing broken PyPI build isolation)
if not os.path.exists('/kaggle/working/BasicSR'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/XPixelGroup/BasicSR.git', '/kaggle/working/BasicSR'], check=True)
    deg_path = '/kaggle/working/BasicSR/basicsr/data/degradations.py'
    if os.path.exists(deg_path):
        with open(deg_path, 'r') as f: content = f.read()
        content = content.replace('from torchvision.transforms.functional_tensor import rgb_to_grayscale', 'from torchvision.transforms.functional import rgb_to_grayscale')
        with open(deg_path, 'w') as f: f.write(content)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '-e', '/kaggle/working/BasicSR'], check=False)

if '/kaggle/working/BasicSR' not in sys.path:
    sys.path.insert(0, '/kaggle/working/BasicSR')

# 2. Setup Real-ESRGAN
if not os.path.exists('/kaggle/working/Real-ESRGAN'):
    subprocess.run(['git', 'clone', 'https://github.com/xinntao/Real-ESRGAN.git', '/kaggle/working/Real-ESRGAN'], check=True)
    try:
        setup_file = '/kaggle/working/Real-ESRGAN/setup.py'
        if os.path.exists(setup_file):
            with open(setup_file, 'r') as sf: s_code = sf.read()
            s_code = s_code.replace("return locals()['__version__']", "loc = {}; exec(compile(open(version_file).read(), version_file, 'exec'), loc); return loc.get('__version__', '0.3.0')")
            with open(setup_file, 'w') as sf: sf.write(s_code)
        subprocess.run([sys.executable, 'setup.py', 'develop'], cwd='/kaggle/working/Real-ESRGAN', check=False)
    except Exception as _e:
        print(f'Notice during Real-ESRGAN setup: {_e}')

if '/kaggle/working/Real-ESRGAN' not in sys.path:
    sys.path.insert(0, '/kaggle/working/Real-ESRGAN')

os.environ['PYTHONPATH'] = f"/kaggle/working/BasicSR:/kaggle/working/Real-ESRGAN:{os.environ.get('PYTHONPATH', '')}"

weights_dir = '/kaggle/working/Real-ESRGAN/weights'
os.makedirs(weights_dir, exist_ok=True)
if not os.path.exists(f'{weights_dir}/realesr-animevideov3.pth'):
    subprocess.run(['wget', '-q', '-O', f'{weights_dir}/realesr-animevideov3.pth', 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth'])

import importlib.util
basicsr_spec = importlib.util.find_spec('basicsr')
if basicsr_spec is not None and basicsr_spec.origin:
    deg_path = os.path.join(os.path.dirname(basicsr_spec.origin), 'data', 'degradations.py')
    if os.path.exists(deg_path):
        with open(deg_path, 'r') as f: content = f.read()
        content = content.replace('from torchvision.transforms.functional_tensor import rgb_to_grayscale', 'from torchvision.transforms.functional import rgb_to_grayscale')
        with open(deg_path, 'w') as f: f.write(content)

print('✅ Real-ESRGAN CUDA Engine Ready (2K/4K Native Acceleration)!')


In [ ]:
print('=== 3. Setting up SeedVC (Bulletproof Voice Cloning Engine) ===')
import subprocess, sys, shutil, os

if not os.path.exists('/kaggle/working/Wan2GP'):
    subprocess.run(['git', 'clone', 'https://github.com/DeepBeepMeep/Wan2GP.git', '/kaggle/working/Wan2GP'], check=True)
    # Check out pinned stable commit to avoid breaking API changes in master branch
    subprocess.run(['git', '-C', '/kaggle/working/Wan2GP', 'checkout', '-q', '6479db36bdc2619a904a852bba9c2d78e1a83f82'], check=True)
    req_file = '/kaggle/working/Wan2GP/requirements.txt'
    with open(req_file, 'r') as rf:
        reqs = rf.read()
    with open(req_file, 'w') as wf:
        for line in reqs.split('\n'):
            if not line.startswith('numpy') and not line.startswith('scipy') and not line.startswith('transformers'):
                wf.write(line + '\n')
        wf.write('transformers==5.19.0\n')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd='/kaggle/working/Wan2GP')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.19.0'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
    try:
        from transformers import PreTrainedModel
        print('✅ transformers.PreTrainedModel verified successfully!')
    except Exception as _e:
        print(f'⚠️ Notice on PreTrainedModel test import: {_e}')

# === ⚡ INSTANT MODEL CACHE CHECK (0-second mount) ===
cached_dir = None
for cand in ['/kaggle/input/seedvc-master-models', '/kaggle/input/seedvc_models_cache']:
    if os.path.exists(cand):
        cached_dir = cand
        break
if not cached_dir:
    import glob
    for cand in glob.glob('/kaggle/input/*seedvc*'):
        if os.path.isdir(cand):
            cached_dir = cand
            break

if cached_dir:
    print(f"⚡ INSTANT LOAD: Found pre-cached models in '{cached_dir}'! Linking in 0 seconds...")
    for root, dirs, files in os.walk(cached_dir):
        for f in files:
            if f.endswith(('.pth', '.pt', '.bin', '.ckpt', '.yaml', '.json', '.safetensors')):
                rel = os.path.relpath(os.path.join(root, f), cached_dir)
                for base in ['models', 'ckpts']:
                    target = os.path.join('/kaggle/working/Wan2GP', base, rel)
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    if not os.path.exists(target):
                        try:
                            os.symlink(os.path.join(root, f), target)
                        except Exception:
                            shutil.copy2(os.path.join(root, f), target)
    print("✅ All SeedVC, Hubert, Whisper & RoFormer models mounted in 0 seconds from Kaggle Dataset!")
else:
    print("🌐 Pre-cached dataset not found — downloading models from HuggingFace (fallback)...")
    os.environ['HF_HOME'] = '/kaggle/tmp/hf_home'
    from huggingface_hub import hf_hub_download, snapshot_download

    def download_model(repo, filename, subfolder):
        for base_dir in ['models', 'ckpts']:
            dest_dir = f'/kaggle/working/Wan2GP/{base_dir}/{subfolder}'
            os.makedirs(dest_dir, exist_ok=True)
            dest_path = f'{dest_dir}/{filename}'
            if not os.path.exists(dest_path):
                print(f'Downloading {filename} to {base_dir}...')
                actual = hf_hub_download(repo_id=repo, filename=f'{subfolder}/{filename}')
                try:
                    os.symlink(actual, dest_path)
                except Exception:
                    shutil.copy(actual, dest_path)

    # Clean official and stable model downloads (from DeepBeepMeep/LTX-2)
    download_model('DeepBeepMeep/LTX-2', 'ar_base.pth', 'seed-vc/v2')
    download_model('DeepBeepMeep/LTX-2', 'cfm_small.pth', 'seed-vc/v2')
    download_model('DeepBeepMeep/LTX-2', 'campplus_cn_common.bin', 'seed-vc')
    download_model('DeepBeepMeep/LTX-2', 'bigvgan_generator.pt', 'bigvgan_v2_22khz_80band_256x')
    download_model('DeepBeepMeep/LTX-2', 'config.json', 'bigvgan_v2_22khz_80band_256x')
    download_model('DeepBeepMeep/LTX-2', 'bsq2048_light.pth', 'seed-vc/bsq2048')
    download_model('DeepBeepMeep/LTX-2', 'bsq32_light.pth', 'seed-vc/bsq32')

    hubert_dir = '/kaggle/working/Wan2GP/models/hubert-large-ll60k'
    if not os.path.exists(hubert_dir):
        print('Downloading Hubert Large model...')
        try:
            snapshot_download(repo_id="facebook/hubert-large-ll60k", local_dir=hubert_dir)
        except Exception as e:
            print(f"Error downloading hubert: {e}")

    whisper_dir = '/kaggle/working/Wan2GP/models/whisper-small'
    if not os.path.exists(whisper_dir):
        print('Downloading Whisper Small model...')
        snapshot_download(repo_id="openai/whisper-small", local_dir=whisper_dir)

    dest_dir = '/kaggle/working/Wan2GP/ckpts/roformer'
    os.makedirs(dest_dir, exist_ok=True)
    if not os.path.exists(f'{dest_dir}/model_bs_roformer_ep_317_sdr_12.9755.ckpt'):
        print('Downloading Roformer...')
        actual = hf_hub_download(repo_id='DeepBeepMeep/Wan2.1', filename='roformer/model_bs_roformer_ep_317_sdr_12.9755.ckpt')
        os.symlink(actual, f'{dest_dir}/model_bs_roformer_ep_317_sdr_12.9755.ckpt')
        actual2 = hf_hub_download(repo_id='DeepBeepMeep/Wan2.1', filename='roformer/model_bs_roformer_ep_317_sdr_12.9755.yaml')
        os.symlink(actual2, f'{dest_dir}/model_bs_roformer_ep_317_sdr_12.9755.yaml')

# --- numpy consistency guard ---
import sys as _sys, subprocess as _sp, importlib as _il
def _sync_numpy_with_loaded():
    try:
        if 'numpy' not in _sys.modules:
            return
        import importlib.metadata as _md
        _il.invalidate_caches()
        loaded = _sys.modules['numpy'].__version__
        disk = _md.version('numpy')
        if loaded != disk:
            print(f"⚠️ numpy on disk ({disk}) != loaded ({loaded}). Reinstalling numpy=={loaded}...")
            _sp.run([_sys.executable, '-m', 'pip', 'install', '-q', f'numpy=={loaded}'], check=True)
            print(f"✅ numpy on disk matched to kernel's loaded version ({loaded}).")
    except Exception as _e:
        print(f"numpy guard notice: {_e}")

_sync_numpy_with_loaded()
print('✅ SeedVC & Model Weights Fully Loaded and Ready!')
# --- Final Verification of SeedVC & MMGP imports ---
if '/kaggle/working/Wan2GP' not in sys.path:
    sys.path.insert(0, '/kaggle/working/Wan2GP')
try:
    from mmgp import offload
    from postprocessing.seedvc.wgp_bridge import convert_audio_file
    print('✅ SeedVC & MMGP & Optimum Quanto verified and ready without torchao errors!')
except Exception as _e:
    print(f'❌ Verification check failed: {_e}')
    raise _e


In [ ]:
print('=== 4. Batch Queue Pipeline Engine ===')
import subprocess, sys, json, time as _time, os
import tempfile, traceback, shutil, re
from scenedetect import detect, ContentDetector
import librosa
import numpy as np

# ============ HELPER FUNCTIONS ============
def extract_best_bgm_segment(bgm_path, target_duration):
    y, sr = librosa.load(bgm_path, sr=22050)
    duration = len(y) / sr
    if duration <= target_duration:
        return 0.0
    rms = librosa.feature.rms(y=y)[0]
    fps = sr / 512
    win = int(target_duration * fps)
    if win == 0 or win > len(rms):
        return 0.0
    avg = np.convolve(rms, np.ones(win)/win, mode='valid')
    return np.argmax(avg) / fps

def run_cmd(cmd):
    try:
        subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(f"\n❌ Command Error Output:\n{e.stderr}\n")
        raise e

def seedvc_voice_clone(input_audio, ref_voice, output_audio, temp_dir, diffusion_steps=12):
    """Robust, leak-proof SeedVC voice conversion wrapper."""
    orig_cwd = os.getcwd()
    sys.path.insert(0, '/kaggle/working/Wan2GP')
    os.chdir('/kaggle/working/Wan2GP')
    try:
        from preprocessing.extract_vocals import extract_vocal_and_background_stems
        from postprocessing.seedvc.wgp_bridge import convert_audio_file, _merge_audio_files_to_wav
        from shared.utils import files_locator as fl

        tag = os.path.splitext(os.path.basename(input_audio))[0]
        vocals_path = os.path.join(temp_dir, f'vc_vocals_{tag}.wav')
        bg_path = os.path.join(temp_dir, f'vc_bg_{tag}.wav')

        vocal_src, bg_src = extract_vocal_and_background_stems(input_audio, vocals_path, bg_path)

        fl.set_checkpoints_paths(['models', 'ckpts', '.'])
        converted_path = os.path.join(temp_dir, f'vc_converted_{tag}.wav')

        # Fast and high-fidelity diffusion_steps=12
        convert_audio_file(vocal_src, ref_voice, converted_path,
                           persistent_models=True, profile_no=4, verbose_level=0,
                           diffusion_steps=diffusion_steps, cfg_rate=0.5, mode=3)

        if bg_src and os.path.exists(bg_src):
            _merge_audio_files_to_wav([converted_path, bg_src], output_audio)
        else:
            shutil.copy2(converted_path, output_audio)
    finally:
        os.chdir(orig_cwd)
        if '/kaggle/working/Wan2GP' in sys.path:
            sys.path.remove('/kaggle/working/Wan2GP')

# ============ SINGLE FILE PROCESSOR ============
def process_single_file(video_path, ref_voice_path, bgm_path, file_index, total_files, outro_vid_path=None, outro_position='end', target_resolution='2k'):
    _has_outro = bool(outro_vid_path)
    prefix = f"[File {file_index+1}/{total_files}]"
    print(f"\n{'='*60}")
    print(f"🎬 {prefix} Starting: {os.path.basename(video_path)}")
    print(f"{'='*60}")

    temp_dir = tempfile.mkdtemp()
    orig_cwd = os.getcwd()
    os.chdir(temp_dir)

    try:
        # 1. Zoom Crop & Trim
        print(f"{prefix} ✂️ Step 1/6: Removing Watermarks & Outro...")
        cropped_vid = 'step1_cropped.mp4'
        duration = float(subprocess.check_output(
            f"ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 '{video_path}'",
            shell=True).decode().strip())
        new_duration = max(0, duration - 3.0)
        zoom_filter = 'scale=trunc(iw*1.08/2)*2:trunc(ih*1.08/2)*2,crop=floor(iw/1.08/2)*2:floor(ih/1.08/2)*2'
        run_cmd(f"ffmpeg -i '{video_path}' -t {new_duration} -vf '{zoom_filter}' -c:v h264_nvenc -preset p4 -b:v 8M '{cropped_vid}' -y")

        # 2. Extract Audio
        print(f"{prefix} 🔊 Step 2/6: Extracting Audio...")
        extracted_audio = 'step2_original_audio.wav'
        run_cmd(f"ffmpeg -i '{cropped_vid}' -q:a 0 -map a '{extracted_audio}' -y")

        # 3. SeedVC Voice Cloning
        print(f"{prefix} 🎙️ Step 3/6: Bulletproof SeedVC Voice Cloning...")
        final_voice = f'{temp_dir}/step3_final_voice.wav'
        seedvc_voice_clone(f'{temp_dir}/{extracted_audio}', ref_voice_path, final_voice, temp_dir, diffusion_steps=12)
        print(f"{prefix} ✅ SeedVC voice cloned successfully!")

        # 4. Scene Transitions
        print(f"{prefix} 🎬 Step 4/6: Adding Cinematic Transitions...")
        transitioned_vid = 'step4_transitioned.mp4'
        scene_list = detect(cropped_vid, ContentDetector(threshold=27.0, min_scene_len=18))
        valid_scenes = [s for s in scene_list if (s[1].get_seconds() - s[0].get_seconds()) > 0.7]

        if len(valid_scenes) > 1:
            fade_dur = 0.25
            filter_str = ""
            for i, scene in enumerate(valid_scenes):
                st = scene[0].get_seconds()
                et = scene[1].get_seconds()
                d = et - st
                added = 0.0
                if i > 0: added += fade_dur/2
                if i < len(valid_scenes)-1: added += fade_dur/2
                sf = (d + added) / d
                filter_str += f"[0:v]trim=start={st}:end={et},setpts={sf}*(PTS-STARTPTS)[v{i}];\n"

            d0 = valid_scenes[0][1].get_seconds() - valid_scenes[0][0].get_seconds()
            offset = d0 + fade_dur/2 - fade_dur
            last = "[v0]"
            for i in range(1, len(valid_scenes)):
                nxt = f"[vout{i}]" if i < len(valid_scenes)-1 else "[outv]"
                filter_str += f"{last}[v{i}]xfade=transition=fade:duration={fade_dur}:offset={offset}{nxt};\n"
                last = nxt
                di = valid_scenes[i][1].get_seconds() - valid_scenes[i][0].get_seconds()
                ai = fade_dur if i < len(valid_scenes)-1 else fade_dur/2
                offset += di + ai - fade_dur

            with open("filter.txt", "w") as f:
                f.write(filter_str.strip().rstrip(';'))
            run_cmd(f"ffmpeg -hwaccel cuda -i '{cropped_vid}' -filter_complex_script filter.txt -map '[outv]' -c:v h264_nvenc -preset p4 -tune hq -b:v 8M '{transitioned_vid}' -y")
        else:
            run_cmd(f"ffmpeg -hwaccel cuda -i '{cropped_vid}' -an -c:v h264_nvenc -preset p4 -tune hq '{transitioned_vid}' -y")

        # 5. BGM Mixing
        print(f"{prefix} 🎵 Step 5/6: Mixing BGM...")
        mixed_audio = 'step5_mixed_audio.wav'
        if bgm_path and os.path.exists(bgm_path):
            best_start = extract_best_bgm_segment(bgm_path, new_duration)
            run_cmd(f"ffmpeg -i '{final_voice}' -ss {best_start:.3f} -i '{bgm_path}' "
                    f"-filter_complex '[1:a]volume=0.09[bg];[0:a][bg]amix=inputs=2:duration=first[aout]' "
                    f"-map '[aout]' '{mixed_audio}' -y")
        else:
            mixed_audio = final_voice

# 6. High-Speed AI Upscaling (2K / 4K) + YouTube Minimum-Compression Assembly
        mode = str(target_resolution).lower().strip()
        is_4k = (mode == '4k')
        target_w, target_h = (2160, 3840) if is_4k else (1440, 2560)
        target_br = "40M" if is_4k else "20M"
        res_label = "4K UHD (2160p)" if is_4k else "2K QHD (1440p)"
        print(f"{prefix} ✨ Step 6/6: High-Speed CUDA {res_label} Upscaling & YouTube-Optimized Assembly...")

        upscaled_vid = 'step6_upscaled.mp4'
        esrgan_script = '/kaggle/working/Real-ESRGAN/inference_realesrgan_video.py'

        transitioned_with_audio = 'step4_with_audio.mp4'
        run_cmd(f"ffmpeg -i '{transitioned_vid}' -f lavfi -i anullsrc=channel_layout=stereo:sample_rate=48000 -c:v copy -c:a aac -shortest '{transitioned_with_audio}' -y")

        # Optimize inference_realesrgan_video preset to ultrafast
        if os.path.exists(esrgan_script):
            with open(esrgan_script, 'r') as f: code = f.read()
            if "preset='ultrafast'" not in code and "vcodec='libx264'" in code:
                code = code.replace("vcodec='libx264'", "vcodec='libx264', preset='ultrafast'")
                with open(esrgan_script, 'w') as f: f.write(code)

        import gc, torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        gc.collect()

        abs_input = os.path.join(temp_dir, transitioned_with_audio)
        upscale_env = os.environ.copy()
        upscale_env['PYTHONPATH'] = f"/kaggle/working/BasicSR:/kaggle/working/Real-ESRGAN:{upscale_env.get('PYTHONPATH', '')}"
        upscale_cmd = [sys.executable, '-u', esrgan_script, '-i', abs_input, '-o', temp_dir,
                      '-n', 'realesr-animevideov3', '-s', '2', '--suffix', 'out', '--tile', '256']
        try:
            subprocess.run(upscale_cmd, cwd='/kaggle/working/Real-ESRGAN', env=upscale_env, check=True, capture_output=True, text=True)
        except subprocess.CalledProcessError as e:
            print(f"❌ ESRGAN Error:\nSTDOUT:\n{e.stdout}\nSTDERR:\n{e.stderr}")
            raise e

        raw_up = os.path.join(temp_dir, transitioned_with_audio.replace('.mp4', '_out.mp4'))
        abs_up = os.path.join(temp_dir, upscaled_vid)
        shutil.move(raw_up, abs_up)

        # High-Fidelity YouTube Minimum-Compression Encode (FastStart, Closed GOP 15, -bf 2, yuv420p, 48kHz AAC 320k)
        main_merged = f'{temp_dir}/main_merged.mp4'
        original_name = os.path.splitext(os.path.basename(video_path))[0]
        outro_tag = f'_OUTRO_{outro_position.upper()}' if _has_outro else ''
        final_output = f'/kaggle/working/{original_name}{outro_tag}.mp4'

        run_cmd(f"ffmpeg -i '{abs_up}' -i '{mixed_audio}' -map 0:v:0 -map 1:a:0 "
                f"-vf 'scale={target_w}:{target_h}:force_original_aspect_ratio=decrease:flags=lanczos,pad={target_w}:{target_h}:(ow-iw)/2:(oh-ih)/2' "
                f"-c:v libx264 -preset fast -profile:v high -level 5.2 -b:v {target_br} -maxrate {int(target_br[:-1])*1.2:.0f}M -bufsize {int(target_br[:-1])*2}M "
                f"-g 15 -keyint_min 15 -sc_threshold 0 -bf 2 -pix_fmt yuv420p -movflags +faststart "
                f"-c:a aac -b:a 320k -ar 48000 '{main_merged}' -y")

        # === OUTRO CONCAT (if provided) ===
        if outro_vid_path and os.path.exists(outro_vid_path):
            print(f"{prefix} 🎙️ Step 7: Processing Outro Clip & SeedVC Voice Clone...")
            outro_raw_audio = f'{temp_dir}/outro_raw.wav'
            run_cmd(f"ffmpeg -i '{outro_vid_path}' -q:a 0 -map a '{outro_raw_audio}' -y")

            outro_cloned_audio = f'{temp_dir}/outro_cloned.wav'
            seedvc_voice_clone(outro_raw_audio, ref_voice_path, outro_cloned_audio, temp_dir, diffusion_steps=12)
            print(f"{prefix} ✅ Outro voice cloned with SeedVC!")

            probe_w = subprocess.check_output(f"ffprobe -v error -select_streams v:0 -show_entries stream=width -of csv=p=0 '{main_merged}'", shell=True).decode().strip()
            probe_h = subprocess.check_output(f"ffprobe -v error -select_streams v:0 -show_entries stream=height -of csv=p=0 '{main_merged}'", shell=True).decode().strip()
            main_w, main_h = int(probe_w), int(probe_h)

            main_fps = subprocess.check_output(f"ffprobe -v error -select_streams v:0 -show_entries stream=r_frame_rate -of csv=p=0 '{main_merged}'", shell=True).decode().strip()
            print(f"📐 Main video: {main_w}x{main_h} @ {main_fps} fps — matching outro to this")

            outro_scaled = f'{temp_dir}/outro_scaled.mp4'
            run_cmd(f"ffmpeg -i '{outro_vid_path}' -i '{outro_cloned_audio}' -map 0:v -map 1:a -vf 'scale={main_w}:{main_h}:force_original_aspect_ratio=decrease,pad={main_w}:{main_h}:(ow-iw)/2:(oh-ih)/2,fps={main_fps}' -c:v h264_nvenc -preset p4 -b:v 8M -pix_fmt yuv420p -c:a aac -ar 44100 '{outro_scaled}' -y")

            concat_list = f'{temp_dir}/concat_outro.txt'
            with open(concat_list, 'w') as cf:
                cf.write(f"file '{main_merged}'\n")
                cf.write(f"file '{outro_scaled}'\n")
            run_cmd(f"ffmpeg -f concat -safe 0 -i '{concat_list}' -c copy '{final_output}' -y")
            print(f"{prefix} ✅ Outro attached!")
        else:
            shutil.move(main_merged, final_output)

        print(f"✅ {prefix} DONE → {final_output}")
        return final_output, None

    except Exception as e:
        traceback.print_exc()
        return None, str(e)
    finally:
        os.chdir('/kaggle/working')

print('✅ Pipeline engine loaded (process_single_file ready).')


In [ ]:
print('=== 5. Run Mode Switch ===')
# Set this before running:
#   'upload' -> launches the Gradio uploader (interactive session, laptop open,
#               used to sync your videos into a Kaggle Dataset).
#   'batch'  -> runs the headless batch processor (used for Save & Run All /
#               Commit background runs — no browser needed, processes all videos
#               sequentially with your laptop closed!).
RUN_MODE = 'batch'
print(f"RUN_MODE = '{RUN_MODE}'")

In [ ]:
print('=== 6. Upload & Auto-Sync to Kaggle Dataset (run this in an interactive session with RUN_MODE=\'upload\') ===')
import gradio as gr
import glob as _glob, os, shutil, json, re

if RUN_MODE != 'upload':
    print("⏭️ RUN_MODE is not 'upload' — skipping the uploader cell.")
else:
    _secrets_ok = False
    try:
        from kaggle_secrets import UserSecretsClient
        _secrets = UserSecretsClient()
        try:
            os.environ['KAGGLE_API_TOKEN'] = _secrets.get_secret('KAGGLE_API_TOKEN')
            _secrets_ok = True
            print("🔑 Kaggle API token loaded (KAGGLE_API_TOKEN).")
        except Exception:
            os.environ['KAGGLE_USERNAME'] = _secrets.get_secret('KAGGLE_USERNAME')
            os.environ['KAGGLE_KEY'] = _secrets.get_secret('KAGGLE_KEY')
            _secrets_ok = True
            print(f"🔑 Kaggle API credentials loaded for user: {os.environ['KAGGLE_USERNAME']}")
    except Exception as e:
        _secrets_ok = False
        print(f"⚠️ Could not load Kaggle API credentials ({e}).")

    import kaggle as _kaggle_module

    STAGING_DIR = '/kaggle/working/dataset_staging'
    os.makedirs(os.path.join(STAGING_DIR, 'videos'), exist_ok=True)
    os.makedirs(os.path.join(STAGING_DIR, 'outros'), exist_ok=True)

    def sync_to_kaggle_dataset(video_files, ref_voice, bgm_audio, outro_vid, cooldown_secs, dataset_slug, kaggle_username):
        if not _secrets_ok:
            return "❌ Kaggle API credentials not loaded. Add KAGGLE_API_TOKEN in Add-ons -> Secrets first."
        if not video_files or len(video_files) == 0:
            return "❌ Please upload at least one video."
        if not ref_voice:
            return "❌ Please upload a reference voice (required for SeedVC)."
        if not dataset_slug or not dataset_slug.strip():
            return "❌ Please enter a dataset name."
        if not kaggle_username or not kaggle_username.strip():
            return "❌ Please enter your Kaggle username."

        slug = re.sub(r'[^a-zA-Z0-9-]', '-', dataset_slug.strip()).lower()

        for f in _glob.glob(os.path.join(STAGING_DIR, 'videos', '*')):
            os.remove(f)
        for f in _glob.glob(os.path.join(STAGING_DIR, 'outros', '*')):
            os.remove(f)
        for name in os.listdir(STAGING_DIR):
            p = os.path.join(STAGING_DIR, name)
            if os.path.isfile(p):
                os.remove(p)

        for vf in video_files:
            src_path = vf.name if hasattr(vf, 'name') else vf
            shutil.copy2(src_path, os.path.join(STAGING_DIR, 'videos', os.path.basename(src_path)))
        shutil.copy2(ref_voice, os.path.join(STAGING_DIR, 'voice_ref.wav'))
        if bgm_audio:
            shutil.copy2(bgm_audio, os.path.join(STAGING_DIR, 'bgm.mp3'))
        if outro_vid:
            outro_list = outro_vid if isinstance(outro_vid, list) else [outro_vid]
            for i, ov in enumerate(outro_list):
                src_path = ov.name if hasattr(ov, 'name') else ov
                ext = os.path.splitext(src_path)[1]
                shutil.copy2(src_path, os.path.join(STAGING_DIR, 'outros', f'outro_{i}{ext}'))

        cfg = {"cooldown_secs": int(cooldown_secs)}
        with open(os.path.join(STAGING_DIR, 'config.json'), 'w') as f:
            json.dump(cfg, f, indent=2)

        api = _kaggle_module.api
        dataset_id = f"{kaggle_username.strip()}/{slug}"
        metadata = {"title": slug, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}
        with open(os.path.join(STAGING_DIR, 'dataset-metadata.json'), 'w') as f:
            json.dump(metadata, f)

        try:
            api.dataset_status(dataset_id)
            exists = True
        except Exception:
            exists = False

        try:
            if exists:
                api.dataset_create_version(STAGING_DIR, version_notes='Auto-sync from Gradio uploader', dir_mode='zip')
                msg = f"✅ Updated existing dataset: {dataset_id} (new version pushed)."
            else:
                api.dataset_create_new(STAGING_DIR, dir_mode='zip', public=False)
                msg = f"✅ Created new private dataset: {dataset_id}."
        except Exception as e:
            return f"❌ Kaggle API upload failed: {e}"

        return (
            f"{msg}\n\n"
            f"Next steps:\n"
            f"1. Sidebar -> Input -> '+ Add Input' -> attach '{slug}'.\n"
            f"2. Change RUN_MODE to 'batch' in Cell 5.\n"
            f"3. Click 'Save Version' -> 'Save & Run All (Commit)'.\n"
            f"4. Laptop closed! Batch runner processes {len(video_files)} short(s) headlessly."
        )

    with gr.Blocks(theme=gr.themes.Base(), title="Batch Shorts Uploader") as uploader_app:
        gr.Markdown('# 📤 Upload & Sync Batch Shorts to Kaggle Dataset')
        with gr.Row():
            with gr.Column():
                videos_in = gr.File(label='1. 📁 Upload ALL Videos (Multiple Shorts)', file_count='multiple', file_types=['video'], type='filepath')
                ref_voice_in = gr.Audio(label='2. 🎙️ Reference Voice (SeedVC Voice Cloning)', type='filepath')
                bgm_in = gr.Audio(label='3. 🎵 Background Music (Optional)', type='filepath')
                outro_in = gr.File(label='4. 🎬 Outro Clips (Optional)', file_count='multiple', file_types=['video'], type='filepath')
                cooldown_sl = gr.Slider(minimum=10, maximum=120, value=30, step=5, label='5. ⏱️ Cooldown Between Files (seconds)')
                dataset_name_tb = gr.Textbox(value='shorts-batch-1', label='6. Kaggle Dataset Name')
                username_tb = gr.Textbox(label='7. Kaggle Username')
                sync_btn = gr.Button('☁️ Sync to Kaggle Dataset', variant='primary', size='lg')
            with gr.Column():
                sync_status = gr.Textbox(label='Sync Status & Next Steps', lines=10)

        sync_btn.click(
            sync_to_kaggle_dataset,
            inputs=[videos_in, ref_voice_in, bgm_in, outro_in, cooldown_sl, dataset_name_tb, username_tb],
            outputs=[sync_status]
        )

    uploader_app.queue()
    uploader_app.launch(share=True)

In [ ]:
print('=== 7. Batch Runner (Headless — no browser needed, safe for background/commit runs) ===')
if RUN_MODE != 'batch':
    print("⏭️ RUN_MODE is not 'batch' — skipping the batch runner cell.")
else:
    import glob, json as _json, zipfile, traceback as _tb, time as _time, os, shutil, random

    def _find_dataset_root():
        candidates = [d for d in glob.glob('/kaggle/input/*') if os.path.isdir(d)]
        if not candidates:
            raise FileNotFoundError(
                "No dataset found under /kaggle/input/. Attach your dataset via "
                "'+ Add Input' in the sidebar before running this cell."
            )

        video_exts = ('.mp4', '.mov', '.mkv', '.webm')
        all_videos_by_root = {}
        for c in candidates:
            found_here = []
            for dirpath, _dirnames, filenames in os.walk(c):
                for fn in filenames:
                    if fn.lower().endswith(video_exts):
                        found_here.append(os.path.join(dirpath, fn))
            all_videos_by_root[c] = found_here
            print(f"   - {os.path.basename(c)}: {len(found_here)} video file(s)")

        best = max(all_videos_by_root, key=lambda k: len(all_videos_by_root[k]))
        if not all_videos_by_root[best]:
            raise FileNotFoundError("None of the attached inputs contain video files.")
        print(f"✅ Using dataset: {best} ({len(all_videos_by_root[best])} videos found)")
        return best

    def _find_one(root, names):
        for name in names:
            p = os.path.join(root, name)
            if os.path.exists(p):
                return p
        for dirpath, _dirnames, filenames in os.walk(root):
            for name in names:
                if name in filenames:
                    return os.path.join(dirpath, name)
        return None

    DATASET_ROOT = _find_dataset_root()
    print(f"📂 Dataset Root: {DATASET_ROOT}")

    video_exts_tuple = ('.mp4', '.mov', '.mkv', '.webm')
    raw_videos = []
    for dirpath, _dirnames, filenames in os.walk(DATASET_ROOT):
        # Don't pick up outros as main videos
        if os.path.basename(dirpath).lower() == 'outros':
            continue
        for fn in filenames:
            if fn.lower().endswith(video_exts_tuple):
                raw_videos.append(os.path.join(dirpath, fn))
    raw_videos = sorted(raw_videos)
    print(f"🎬 Found {len(raw_videos)} raw short video(s): {[os.path.basename(v) for v in raw_videos]}")

    ref_voice = _find_one(DATASET_ROOT, ['voice_ref.wav', 'voice_ref.mp3', 'reference_voice.wav', 'global_voice.wav', 'global_voice.mp3'])
    bgm_audio = _find_one(DATASET_ROOT, ['bgm.mp3', 'bgm.wav', 'music.mp3', 'global_bgm.mp3'])
    outro_vids = []
    for dirpath, _dirnames, filenames in os.walk(DATASET_ROOT):
        if os.path.basename(dirpath).lower() in ['outros', 'outro']:
            for fn in filenames:
                if fn.lower().endswith(video_exts_tuple):
                    outro_vids.append(os.path.join(dirpath, fn))
    if not outro_vids:
        single_outro = _find_one(DATASET_ROOT, ['outro.mp4', 'outro.mov', 'outro.mkv', 'promo.mp4', 'global_outro.mp4'])
        if single_outro: outro_vids.append(single_outro)
    outro_vid = outro_vids[0] if outro_vids else None
    config_path = _find_one(DATASET_ROOT, ['config.json'])

    print(f"🎙️ Reference voice: {ref_voice}")
    print(f"🎶 BGM: {bgm_audio}")
    print(f"🎬 Outro clips found: {len(outro_vids)}")

    if not raw_videos:
        raise FileNotFoundError("No raw videos found in dataset.")
    if not ref_voice:
        raise FileNotFoundError("No reference voice found (expected voice_ref.wav).")

    cfg = {"cooldown_secs": 30}
    if config_path:
        with open(config_path) as f:
            cfg.update(_json.load(f))
        print(f"⚙️ Loaded config: {cfg}")

    OUTPUT_DIR = '/kaggle/working/outputs'
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    total = len(raw_videos)
    results = []
    batch_start = _time.time()
    cooldown = int(cfg.get("cooldown_secs", 30))

    # Calculate outro placement
    _midroll_indices = set()
    if outro_vid:
        if total <= 5:
            _outro_idx_list = [i for i in range(total) if (i + 1) % 3 == 0]
        else:
            _num = max(1, round(total * 0.4))
            _outro_idx_list = [round((k + 1) * total / (_num + 1)) - 1 for k in range(_num)]
        _midroll_count = min(2, max(1, len(_outro_idx_list) // 2))
        if _outro_idx_list:
            _midroll_indices = set(random.sample(_outro_idx_list, min(_midroll_count, len(_outro_idx_list))))
            print(f"🎯 Mid-roll outros at indices: {[i+1 for i in _midroll_indices]}")

    for idx, raw_vid in enumerate(raw_videos):
        fname = os.path.basename(raw_vid)
        t0 = _time.time()
        print(f"\n{'='*60}\n🎬 [{idx+1}/{total}] Processing: {fname}\n{'='*60}")
        try:
            attach_outro = None
            outro_pos = 'end'
            if outro_vid:
                if total <= 5:
                    is_outro = ((idx + 1) % 3 == 0)
                else:
                    num_outros = max(1, round(total * 0.4))
                    outro_indices = {round((k + 1) * total / (num_outros + 1)) - 1 for k in range(num_outros)}
                    is_outro = (idx in outro_indices)
                attach_outro = random.choice(outro_vids) if (is_outro and outro_vids) else None
                if attach_outro and idx in _midroll_indices:
                    outro_pos = 'mid'

            if attach_outro:
                print(f"🎬 Video #{idx+1}/{total} — OUTRO [{outro_pos.upper()}] attached")

            target_res = str(cfg.get("resolution", "2k")).lower()
            output_path, error = process_single_file(raw_vid, ref_voice, bgm_audio, idx, total, outro_vid_path=attach_outro, outro_position=outro_pos, target_resolution=target_res)
        except Exception as e:
            _tb.print_exc()
            output_path, error = None, str(e)
        elapsed = _time.time() - t0

        if error:
            print(f"❌ [{idx+1}] {fname} — FAILED ({error})")
            results.append({"video": fname, "status": "failed", "message": error})
        else:
            out_name = os.path.basename(output_path)
            final_dest = os.path.join(OUTPUT_DIR, out_name)
            shutil.copy2(output_path, final_dest)
            outro_label = " [+OUTRO]" if attach_outro else ""
            print(f"✅ [{idx+1}] {fname}{outro_label} — Done in {elapsed:.0f}s -> {out_name}")
            results.append({"video": fname, "status": "success", "message": f"Done in {elapsed:.0f}s{outro_label}", "output": final_dest})

        if idx < total - 1:
            print(f"⏸️ Cooldown: {cooldown}s before next file...")
            _time.sleep(cooldown)
            import gc, torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
            gc.collect()

    batch_total = _time.time() - batch_start

    zip_path = '/kaggle/working/all_final_videos.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for r in results:
            if r["status"] == "success":
                zf.write(r["output"], os.path.basename(r["output"]))

    print(f"\n{'='*60}\n📊 BATCH SHORTS REPORT ({batch_total/60:.1f} minutes total)\n{'='*60}")
    success_count = sum(1 for r in results if r["status"] == "success")
    for r in results:
        icon = "✅" if r["status"] == "success" else "❌"
        print(f"  {icon} {r['video']}: {r['message']}")
    print(f"\n{success_count}/{len(results)} shorts completed successfully.")
    print(f"📦 All outputs zipped at: {zip_path}")
